In [3]:
import pandas as pd
df = pd.read_csv("churn.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 27 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7032 non-null   float64
 1   SeniorCitizen                            7032 non-null   int64  
 2   Partner                                  7032 non-null   float64
 3   Dependents                               7032 non-null   float64
 4   tenure                                   7032 non-null   int64  
 5   PhoneService                             7032 non-null   float64
 6   MultipleLines                            7032 non-null   float64
 7   OnlineSecurity                           7032 non-null   float64
 8   OnlineBackup                             7032 non-null   float64
 9   DeviceProtection                         7032 non-null   float64
 10  TechSupport                              7032 no

# Model accuracy before handling imbalanced data

In [4]:
df['Churn'].value_counts()

,count
Churn,
0.0,5163
1.0,1869


In [5]:
from sklearn.model_selection import train_test_split
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.83      0.89      0.86      1033
         1.0       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407



precision, recall and f1-score of class 1 is less compared to class 0

# Method 1: Undersampling

In [11]:
df['Churn'].value_counts()

,count
Churn,
0.0,5163
1.0,1869


In [13]:
# Divide by class
df_class_0 = df[df['Churn']==0]
df_class_1 = df[df['Churn']==1]

In [15]:
# Undersample class 0 and concat the DataFrames of both class
df_class_0 = df_class_0.sample(len(df_class_1))
df_under = pd.concat([df_class_0, df_class_1], axis=0)

In [16]:
df_under['Churn'].value_counts()

,count
Churn,
0.0,1869
1.0,1869


In [21]:
from sklearn.model_selection import train_test_split
X = df_under.drop('Churn', axis=1)
y = df_under['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.77      0.73      0.75       374
         1.0       0.74      0.78      0.76       374

    accuracy                           0.75       748
   macro avg       0.75      0.75      0.75       748
weighted avg       0.75      0.75      0.75       748



Here, precison, recall and f1-score of both the classes are almost close by

# Method 2: Oversampling

In [20]:
df['Churn'].value_counts()

,count
Churn,
0.0,5163
1.0,1869


In [24]:
# Divide by class
df_class_0 = df[df['Churn']==0]
df_class_1 = df[df['Churn']==1]

In [25]:
# Oversample class 1 and concat the DataFrames of both class
df_class_1 = df_class_1.sample(len(df_class_0), replace=True)
df_over = pd.concat([df_class_0, df_class_1], axis=0)

In [26]:
df_over['Churn'].value_counts()

,count
Churn,
0.0,5163
1.0,5163


In [27]:
from sklearn.model_selection import train_test_split
X = df_over.drop('Churn', axis=1)
y = df_over['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.81      0.69      0.75      1033
         1.0       0.73      0.84      0.78      1033

    accuracy                           0.76      2066
   macro avg       0.77      0.76      0.76      2066
weighted avg       0.77      0.76      0.76      2066



# Method 3: SMOTE

In [29]:
df['Churn'].value_counts()

,count
Churn,
0.0,5163
1.0,1869


In [31]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [33]:
!pip install imbalanced-learn

In [35]:
# Sample using SMOTE lib
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy='minority')
X_sm, y_sm = smote.fit_resample(X, y)
y_sm.value_counts()

,count
Churn,
0.0,5163
1.0,5163


In [36]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_sm, y_sm, test_size=0.2, random_state=42, stratify=y_sm)

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.84      0.80      0.82      1033
         1.0       0.81      0.84      0.83      1033

    accuracy                           0.82      2066
   macro avg       0.82      0.82      0.82      2066
weighted avg       0.82      0.82      0.82      2066



# Method 4: Use of Ensemble with undersampling

In [40]:
df['Churn'].value_counts()

,count
Churn,
0.0,5163
1.0,1869


In [41]:
from sklearn.model_selection import train_test_split
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [44]:
y_train.value_counts()

,count
Churn,
0.0,4130
1.0,1495


In [43]:
class_0, class_1 = y_train.value_counts()
class_0, class_1

(4130, 1495)

In [45]:
class_0/class_1

2.762541806020067

model1 --> class1(1495) + class0(0, 1495)

model2 --> class1(1495) + class0(1496, 2990)

model3 --> class1(1495) + class0(2990, 4130

In [46]:
df_en = X_train.copy()
df_en['Churn'] = y_train
df_en.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
1408,1.0,0,1.0,1.0,65,1.0,1.0,1.0,1.0,1.0,...,True,False,False,False,True,False,True,False,False,0.0
6992,1.0,0,0.0,0.0,26,0.0,0.0,0.0,0.0,1.0,...,False,False,True,False,False,False,False,True,False,0.0
3349,0.0,0,1.0,0.0,68,1.0,1.0,0.0,1.0,1.0,...,True,False,False,False,True,False,True,False,False,0.0
4486,1.0,0,0.0,0.0,3,1.0,0.0,0.0,1.0,0.0,...,True,False,True,False,False,False,False,True,False,0.0
3535,0.0,0,1.0,0.0,49,0.0,0.0,1.0,0.0,0.0,...,False,False,True,False,False,True,False,False,False,0.0


In [47]:
df_class_0 = df_en[df_en['Churn']==0]
df_class_1 = df_en[df_en['Churn']==1]

In [51]:
def get_train_batch(df_majority, df_minority, start, end):
  df_train = pd.concat([df_majority[start:end], df_minority], axis=0)
  X_train = df_train.drop('Churn', axis=1)
  y_train = df_train['Churn']
  return X_train, y_train

model1 --> class1(1495) + class0(0, 1495)

In [55]:
X_train, y_train = get_train_batch(df_class_0, df_class_1, 0, 1495)

In [56]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
y_pred1 = lr.predict(X_test)

model2 --> class1(1495) + class0(1495, 2990)

In [59]:
X_train, y_train = get_train_batch(df_class_0, df_class_1, 1495, 2990)

In [60]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
y_pred2 = lr.predict(X_test)

model3 --> class1(1495) + class0(2990, 4130)

In [61]:
X_train, y_train = get_train_batch(df_class_0, df_class_1, 2990, 4130)

In [62]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
y_pred3 = lr.predict(X_test)

In [63]:
y_pred_final = y_pred1.copy()
for i in range(len(y_pred1)):
  n_ones = y_pred1[i] + y_pred2[i] + y_pred3[i]
  if n_ones > 1:
    y_pred_final[i] = 1
  else:
    y_pred_final[i] = 0

In [64]:
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

         0.0       0.91      0.69      0.78      1033
         1.0       0.48      0.81      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.69      1407
weighted avg       0.80      0.72      0.74      1407

